In [47]:
import os
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import cv2
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from scipy.spatial.transform import Rotation as R

# === CONFIGURATION ===
DATA_DIR = "/home/tim-external/ros_ws/src/fsregistration/plotting_results/2d/data"
print(f'Data directory: {DATA_DIR}')
print('Imports OK')

Data directory: /home/tim-external/ros_ws/src/fsregistration/plotting_results/2d/data
Imports OK


In [48]:
def load_csv_array(filepath):
    if not os.path.exists(filepath):
        return None
    return np.loadtxt(filepath)


def load_meta_csv(filepath):
    if not os.path.exists(filepath):
        return None
    return pd.read_csv(filepath)


# Load data
img1 = load_csv_array(os.path.join(DATA_DIR, 'input1.csv'))
img2 = load_csv_array(os.path.join(DATA_DIR, 'input2.csv'))
meta_df = load_meta_csv(os.path.join(DATA_DIR, 'registration_meta.csv'))

if img1 is None or img2 is None or meta_df is None:
    print('ERROR: Missing data files. Make sure viewBoreasPairs.py has been run.')
    print(f'  input1.csv exists: {os.path.exists(os.path.join(DATA_DIR, "input1.csv"))}')
    print(f'  input2.csv exists: {os.path.exists(os.path.join(DATA_DIR, "input2.csv"))}')
    print(f'  registration_meta.csv exists: {os.path.exists(os.path.join(DATA_DIR, "registration_meta.csv"))}')
elif len(meta_df) == 0:
    print('ERROR: registration_meta.csv is empty (save failed).')
else:
    meta = meta_df.iloc[0]
    N = int(meta['N'])
    rot_deg = meta['rot_angle_deg']
    tx_m = meta['tx_m']
    ty_m = meta['ty_m']
    confidence = meta['confidence']
    time_ms = meta['time_ms']
    gt_rot_err = meta['gt_rot_err_deg']
    gt_trans_err = meta['gt_trans_err_m']
    frame1 = int(meta['frame1'])
    frame2 = int(meta['frame2'])
    n_solutions = int(meta['n_solutions'])
    pixel_size = meta['pixel_size_m']
    
    print(f'Loaded pair: frames {frame1}->{frame2}')
    print(f'Image size: {N}x{N}')
    print(f'Pixel size: {pixel_size} m')
    print(f'Rotation: {rot_deg:.2f} deg')
    print(f'Translation: tx={tx_m:.2f} m, ty={ty_m:.2f} m')
    print(f'Confidence: {confidence:.3f}')
    print(f'GT Error: rot={gt_rot_err:.2f} deg, trans={gt_trans_err:.2f} m')
    print(f'Solutions: {n_solutions}')
    print()
    print(meta_df.to_string())

Loaded pair: frames 50->51
Image size: 256x256
Pixel size: 0.78125 m
Rotation: 0.55 deg
Translation: tx=0.42 m, ty=-0.24 m
Confidence: 0.895
GT Error: rot=1.26 deg, trans=1.15 m
Solutions: 0

   frame1  frame2  rot_angle_deg      tx_m     ty_m  confidence    time_ms  gt_rot_err_deg  gt_trans_err_m    N  n_solutions  radius_m  pixel_size_m method
0      50      51       0.547546  0.420798 -0.24455    0.894737  33.142328        1.257113        1.148819  256            0       100       0.78125   sift


## Registration Result

The estimated rotation (`R.from_euler("z", yaw)`) and translation (`[tx, -ty]`) are used to build a 4x4 transform, converted to a 2D image affine via `get_affine_matrix`, then img2 is warped with `cv2.warpPerspective` and blended 50/50 with img1. The same affine transform pipeline is used as in `viewBoreasPairs.py`. The implicit pixel-magnitude bug in `get_affine_matrix` (translation in meters, treated as pixels by cv2) matches the reference script precisely.

In [49]:
if img1 is not None and img2 is not None and meta_df is not None:
    # Build 4x4 transform (same as viewBoreasPairs.py)
    yaw = np.deg2rad(rot_deg)
    transform_4x4 = np.eye(4)
    transform_4x4[:3, :3] = R.from_euler("z", yaw).as_matrix()
    transform_4x4[:3, 3] = [tx_m, -ty_m, 0.0]

    # Convert to 2D image affine (same as get_affine_matrix from boreasDatasetLoader.py)
    inv_t = np.linalg.inv(transform_4x4)
    affine_2d = np.eye(3)
    affine_2d[:2, :2] = inv_t[:2, :2]
    affine_2d[0, 2] = -inv_t[1, 3] / pixel_size
    affine_2d[1, 2] = inv_t[0, 3] / pixel_size
    # Rotation center compensation (rotate around image center)
    c_center = N / 2.0
    T_c = np.eye(3)
    T_c[:2, 2] = [c_center, c_center]
    T_c_inv = np.eye(3)
    T_c_inv[:2, 2] = [-c_center, -c_center]
    affine_2d = T_c @ affine_2d @ T_c_inv

    # Warp img2 using cv2.warpPerspective
    warped = cv2.warpPerspective(img2, affine_2d, (N, N))
    blended = 0.5 * img1 + 0.5 * warped

    # Normalize for display
    img1_norm = (img1 - img1.min()) / (img1.max() - img1.min() + 1e-10)
    img2_norm = (img2 - img2.min()) / (img2.max() - img2.min() + 1e-10)
    blended_norm = (blended - blended.min()) / (blended.max() - blended.min() + 1e-10)

    # Title text
    title = (f'Pair (frames {frame1}->{frame2})<br>'
             f'Rot: {rot_deg:.2f} deg | Tx: {tx_m:.2f} m | Ty: {ty_m:.2f} m<br>'
             f'Confidence: {confidence:.3f} | Time: {time_ms:.1f} ms | Solutions: {n_solutions}<br>'
             f'GT Error: Rot={gt_rot_err:.2f} deg | Trans={gt_trans_err:.2f} m')

    # Show img1 and img2 side by side (both square in 1x2 layout)
    fig = make_subplots(
        rows=1, cols=2,
        subplot_titles=('Input Image 1', 'Input Image 2'),
        specs=[[{'type': 'heatmap'}, {'type': 'heatmap'}]],
        horizontal_spacing=0.1
    )
    fig.add_trace(go.Heatmap(z=img1_norm, colorscale='Viridis', showscale=True,
                             colorbar_title='Intensity'), row=1, col=1)
    fig.add_trace(go.Heatmap(z=img2_norm, colorscale='Viridis', showscale=False), row=1, col=2)
    fig.update_layout(
        height=500,
        width=1000,
        title_text=title,
        title_font_size=14,
        title_x=0.5
    )
    fig.update_xaxes(showticklabels=False, showgrid=False)
    fig.update_yaxes(showticklabels=False, showgrid=False)
    fig.update_yaxes(scaleanchor="x",  scaleratio=1, row=1, col=1)
    fig.update_yaxes(scaleanchor="x2", scaleratio=1, row=1, col=2)
    fig.update_yaxes(scaleanchor="x3", scaleratio=1, row=2, col=1)
    fig.update_yaxes(scaleanchor="x4", scaleratio=1, row=2, col=2)
    fig.show()
else:
    print('Cannot display: missing data files.')


In [50]:
if img1 is not None and img2 is not None and meta_df is not None:
    fig2 = go.Figure(data=go.Heatmap(
        z=blended_norm,
        colorscale='Viridis',
        showscale=True,
        colorbar_title='Intensity'
    ))
    fig2.update_layout(
        title_text=f'Blended (warped overlay) - Pair {frame1}->{frame2}',
        title_x=0.5,
        height=700,
        width=700,
    )
    fig2.update_xaxes(showticklabels=False, showgrid=False)
    fig2.update_yaxes(showticklabels=False, showgrid=False)
    fig.update_yaxes(scaleratio=1)
    fig2.show()
else:
    print('Cannot display: missing data files.')
